In [2]:
from pathlib import Path
import os

import pandas as pd
import plotly.express as px
from IPython.display import display

KUB_TRACE_PATH = Path(os.getenv("KUB_TRACE_PATH", "../../decisions.jsonl")).expanduser().resolve()
default_sim = Path(os.getenv("SIM_TRACE_PATH", "../../kubenergysched/results_latest/decisions.jsonl")).expanduser()
if default_sim.exists():
    SIM_TRACE_PATH = default_sim.resolve()
else:
    candidates = sorted(Path("../../kubenergysched").glob("results_*/decisions.jsonl"))
    SIM_TRACE_PATH = candidates[-1].resolve() if candidates else None

def load_trace(path: Path, label: str) -> pd.DataFrame:
    if path is None or not path.exists():
        raise FileNotFoundError(f"{label} trace not found at {path}")
    df = pd.read_json(path, lines=True)
    df["result_type"] = df["result_type"].fillna(label)
    df["source"] = df["source"].fillna(label)
    df["site"] = df["site"].replace("", "unknown")
    df["scheduler"] = df["scheduler"].fillna("unknown")
    df["result_label"] = df["result_type"].str.replace("_result", "", regex=False)
    df["trace_path"] = str(path)
    return df

frames: list[pd.DataFrame] = []
missing: dict[str, str] = {}
for label, path in ("kub_result", KUB_TRACE_PATH), ("sim_result", SIM_TRACE_PATH):
    try:
        frames.append(load_trace(path, label))
    except FileNotFoundError as exc:
        missing[label] = str(exc)

if not frames:
    raise RuntimeError(f"No traces loaded: {missing}")

df = pd.concat(frames, ignore_index=True)
df["result_type"] = df["result_type"].astype("category")
df.sort_values(["result_type", "job_id"], inplace=True)

if missing:
    print("Sources missing data:")
    for label, msg in missing.items():
        print(f" - {label}: {msg}")

summary = (
    df.groupby(["result_type", "site"], dropna=False)
      .size()
      .reset_index(name="count")
      .sort_values(["result_type", "count"], ascending=[True, False])
)

display(df.head())
summary


/tmp/ipykernel_916104/1627947086.py:49: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(["result_type", "site"], dropna=False)


,result_type,result_id,scheduler,source,job_id,site,node,e_norm,c_norm,cost,theta_e,theta_c,forecast_used,fallback,reject_reason,queued_at,started_at,ended_at,result_label,trace_path
36,kub_result,kub_result_01dae029-6158-4b80-be4e-9d45f5fe778a,engine,kubernetes,01dae029-6158-4b80-be4e-9d45f5fe778a,unknown,,0.0,0.0,0.0,0.5,0.5,False,True,no feasible candidates,0001-01-01T00:00:00Z,0001-01-01T00:00:00Z,0001-01-01T00:00:00Z,kub,/home/goncalo/KubeEnergyScheduler/decisions.jsonl
37,kub_result,kub_result_01dae029-6158-4b80-be4e-9d45f5fe778a,engine,kubernetes,01dae029-6158-4b80-be4e-9d45f5fe778a,unknown,,0.0,0.0,0.0,0.5,0.5,False,True,no feasible candidates,0001-01-01T00:00:00Z,0001-01-01T00:00:00Z,0001-01-01T00:00:00Z,kub,/home/goncalo/KubeEnergyScheduler/decisions.jsonl
2,kub_result,kub_result_0264426d-50b3-4100-ac74-111d0588ee77,engine,kubernetes,0264426d-50b3-4100-ac74-111d0588ee77,unknown,,0.0,0.0,0.0,0.5,0.5,False,True,no feasible candidates,0001-01-01T00:00:00Z,0001-01-01T00:00:00Z,0001-01-01T00:00:00Z,kub,/home/goncalo/KubeEnergyScheduler/decisions.jsonl
20,kub_result,kub_result_0264426d-50b3-4100-ac74-111d0588ee77,engine,kubernetes,0264426d-50b3-4100-ac74-111d0588ee77,unknown,,0.0,0.0,0.0,0.5,0.5,False,True,no feasible candidates,0001-01-01T00:00:00Z,0001-01-01T00:00:00Z,0001-01-01T00:00:00Z,kub,/home/goncalo/KubeEnergyScheduler/decisions.jsonl
15,kub_result,kub_result_03a10c09-b33f-4168-b269-2298d9c5e7fd,engine,kubernetes,03a10c09-b33f-4168-b269-2298d9c5e7fd,unknown,,0.0,0.0,0.0,0.5,0.5,False,True,no feasible candidates,0001-01-01T00:00:00Z,0001-01-01T00:00:00Z,0001-01-01T00:00:00Z,kub,/home/goncalo/KubeEnergyScheduler/decisions.jsonl


,result_type,site,count
1,kub_result,unknown,67
0,kub_result,B,3
3,sim_result,unknown,4000
2,sim_result,B,0


In [ ]:
fig = px.scatter(
    df,
    x="e_norm",
    y="c_norm",
    color="result_type",
    symbol="scheduler",
    hover_data=["job_id", "site", "node", "cost", "fallback", "trace_path"],
    title="Energy vs Carbon normalised cost",
)

buttons = []
for label in ("Both", "Kubernetes", "Simulation"):
    visible = []
    for trace in fig.data:
        if label == "Both":
            visible.append(True)
        elif label == "Kubernetes":
            visible.append(trace.name == "kub_result")
        else:
            visible.append(trace.name == "sim_result")
    buttons.append(
        {
            "label": label,
            "method": "update",
            "args": [{"visible": visible}],
        }
    )

fig.update_layout(
    legend_title_text="Result type",
    updatemenus=[
        {
            "type": "buttons",
            "direction": "left",
            "x": 0.0,
            "y": 1.15,
            "buttons": buttons,
        }
    ],
)
fig.show()


In [ ]:
site_counts = (
    df.groupby(["result_type", "site"], dropna=False)
      .size()
      .reset_index(name="count")
)
site_fig = px.bar(
    site_counts,
    x="site",
    y="count",
    color="result_type",
    barmode="group",
    category_orders={"result_type": sorted(site_counts["result_type"].unique())},
    title="Decisions per site",
)
site_fig.update_layout(xaxis_title="site", yaxis_title="count")
site_fig.show()
